# Dhruv Sharma

# AI Lead Qualification + Smart Response System

## Final Assessment Submission — Dream Reflection Media

Production-ready AI lead qualification workflow demonstrating:
- scalable architecture
- hybrid AI classification
- response generation
- monitoring & observability
- edge case handling
- production-aware engineering decisions

### Objective
Build a production-aware AI workflow that:
- Accepts lead inputs from forms/chat
- Classifies leads into Hot / Warm / Cold
- Generates personalized responses
- Handles failures and edge cases
- Demonstrates scalable architecture thinking

---

## Design Philosophy

This solution prioritizes:
- Clarity over unnecessary complexity
- Production-aware engineering decisions
- Modular architecture
- Scalability and reliability
- Human-like yet controlled AI responses

## 1. High-Level Architecture

```text
Form / Chat Widget
        ↓
 FastAPI API Layer
        ↓
 Input Validation Layer
        ↓
 Lead Classification Engine
        ↓
 Confidence Scoring
        ↓
 Personalized Response Generator
        ↓
 PostgreSQL / CRM Storage
        ↓
 Monitoring + Logging + Alerts
```

## 2. Imports & Setup

The system uses:
- `pydantic` for schema validation
- `typing` for optional fields
- `logging` for observability

In [8]:
from pydantic import BaseModel
from typing import Optional
from pprint import pprint
import logging

logging.basicConfig(level=logging.INFO)


## 3. Lead Input Schema - Classification Logic

This schema represents incoming lead data from:
- Website forms
- Chat widgets
- CRM integrations
- API requests

The schema ensures:
- Structured validation
- Consistent processing
- Cleaner downstream inference


## Hybrid Classification Strategy

This implementation uses:
1. Rule-based scoring
2. Context-aware logic

### Why Hybrid?
A fully LLM-based classifier may:
- hallucinate
- produce inconsistent outputs
- increase latency/cost

A hybrid approach improves:
- reliability
- explainability
- production safety

---

## Classification Factors

### Hot Lead
- High budget
- Clear urgency
- Enterprise intent
- Detailed use case

### Warm Lead
- Medium intent
- Some urgency
- Moderate engagement

### Cold Lead
- Low clarity
- Low urgency
- Generic inquiries

In [9]:
class Lead(BaseModel):
    name: str
    email: str
    company: Optional[str] = ""
    budget: Optional[int] = 0
    urgency: Optional[str] = ""
    message: str


## 4. Hybrid Lead Classification Strategy

This solution uses a hybrid rule-based + AI-oriented scoring approach for reliability and explainability.

In [10]:
def classify_lead(lead):

    score = 0

    # Budget scoring
    if lead.budget >= 10000:
        score += 2
    elif lead.budget >= 5000:
        score += 1

    # Urgency scoring
    urgency = lead.urgency.lower()

    if "urgent" in urgency:
        score += 2
    elif "soon" in urgency:
        score += 1

    # Message quality scoring
    if len(lead.message) > 50:
        score += 1

    # Enterprise intent
    if not lead.email.endswith(("gmail.com", "yahoo.com", "hotmail.com")):
        score += 1

    # Final classification
    if score >= 5:
        return {
            "classification": "Hot",
            "confidence": 0.92
        }

    elif score >= 3:
        return {
            "classification": "Warm",
            "confidence": 0.76
        }

    return {
        "classification": "Cold",
        "confidence": 0.58
    }


## 5. Human Escalation Logic

Low-confidence predictions are routed for manual review.

In [11]:
def requires_human_review(confidence):

    if confidence < 0.60:
        return True

    return False


## 6. Prompt Engineering Layer

### Classification Prompt

```text
You are an AI lead qualification assistant.

Classify the lead as:
- Hot
- Warm
- Cold

Consider:
- urgency
- budget
- purchase intent
- message clarity

Return:
{
  "classification": "",
  "confidence": "",
  "reason": ""
}
```

### Response Generation Prompt


## Goal
Generate:
- Professional
- Personalized
- Context-aware
responses.

The response generator adapts messaging based on:
- Lead classification
- User intent
- Business urgency

---

## Production Considerations
In production, this layer could:
- Use LLMs
- Integrate CRM history
- Inject semantic context
- Support multilingual responses


## 7. Personalized Response Generation

In [12]:
def generate_response(classification, lead):

    base_response = f"""
Hi {lead.name},

Thank you for reaching out regarding your requirements.

"""

    if classification == "Hot":

        return base_response + """
We believe your use case is highly aligned with our solutions.

Our team would love to schedule a priority consultation to discuss your requirements in detail.

We will contact you shortly.

Best Regards,
AI Solutions Team
"""

    elif classification == "Warm":

        return base_response + """
Your requirements look promising, and we would be happy to explore potential solutions with you.

Our team will review your request and connect with you soon.

Best Regards,
AI Solutions Team
"""

    return base_response + """
Thank you for your interest.

We have received your request and our team may reach out if your requirements align with our current offerings.

Best Regards,
AI Solutions Team
"""


## 8. AI Processing Pipeline

## Pipeline Flow

Input → Validation → Classification → Response Generation

---

## Engineering Decisions
This pipeline is:
- Modular
- Easy to scale
- Easy to debug
- Production-friendly

---

## Failure Prevention
The pipeline includes:
- Input validation
- Error handling
- Logging
- Graceful degradation


In [13]:
def process_lead(lead):

    try:

        # Input validation
        if len(lead.message.strip()) < 5:

            return {
                "status": "failed",
                "reason": "Lead message too short"
            }

        # Classification
        result = classify_lead(lead)

        classification = result["classification"]
        confidence = result["confidence"]

        # Human escalation
        if requires_human_review(confidence):

            return {
                "status": "review_required",
                "reason": "Low confidence classification"
            }

        # Response generation
        response = generate_response(classification, lead)

        logging.info(
            f"Lead processed successfully | Classification: {classification}"
        )

        return {
            "status": "success",
            "classification": classification,
            "confidence": confidence,
            "response": response
        }

    except Exception as e:

        logging.error(f"Pipeline failure: {str(e)}")

        return {
            "status": "error",
            "reason": str(e),
            "fallback_response": "Our team will contact you shortly."
        }


## 9. Example Test Run

In [14]:
sample_lead = Lead(
    name="Dhruv Sharma",
    email="dhruv@drm.com",
    company="Dream Reflection Media",
    budget=15000,
    urgency="urgent",
    message="We are looking for an AI automation solution for lead qualification and customer engagement."
)

result = process_lead(sample_lead)

pprint(result)


INFO:root:Lead processed successfully | Classification: Hot


{'classification': 'Hot',
 'confidence': 0.92,
 'response': '\n'
             'Hi Dhruv Sharma,\n'
             '\n'
             'Thank you for reaching out regarding your requirements.\n'
             '\n'
             '\n'
             'We believe your use case is highly aligned with our solutions.\n'
             '\n'
             'Our team would love to schedule a priority consultation to '
             'discuss your requirements in detail.\n'
             '\n'
             'We will contact you shortly.\n'
             '\n'
             'Best Regards,\n'
             'AI Solutions Team\n',
 'status': 'success'}



# 10. Production Architecture (Pseudo-System Design)

## Recommended Architecture

### Input Layer
- Website forms
- Chat widgets
- CRM integrations
- API endpoints

### Backend/API Layer
- FastAPI
- Authentication
- Rate limiting

### Async Processing
- Celery + Redis
OR
- Kafka / RabbitMQ

### Storage
- PostgreSQL
- Redis cache
- Optional vector database

### AI Layer
- LLM classification
- Prompt pipelines
- Context retrieval

### Monitoring
- Prometheus
- Grafana
- Structured logging
- Alerts

---

## Why Async Processing?
Prevents:
- API blocking
- latency spikes
- user-facing delays

Improves:
- scalability
- retry handling
- reliability


## 11. Async Scalability Design

Recommended async processing tools:
- Celery + Redis
- Kafka
- RabbitMQ

Benefits:
- lower latency
- retry handling
- scalable inference
- non-blocking APIs


## 12. Edge Case Handling

## Low / Garbage Input
- Input validation
- Spam filtering
- Minimum text thresholds

## Ambiguous Leads
- Confidence thresholds
- Clarification prompts
- Human review escalation

## Model Failure
- Retry logic
- Rule-based fallback
- Cached response templates

## Timeout / API Failure
- Async queues
- Circuit breakers
- Graceful degradation

## Incorrect Classification
- Feedback loops
- Retraining pipelines
- Human override system


## 13. Monitoring & Observability

## Metrics Tracked
- API latency
- Failure rate
- Queue backlog
- Token usage
- Classification confidence
- Response quality

## Tools
- Prometheus
- Grafana
- ELK Stack
- OpenTelemetry

## Alerting
Alerts trigger on:
- increased failure rate
- queue overflow
- latency spikes
- abnormal response quality



## 14. Trade-Offs

## Trade-Off 1 — Hybrid Classification
Using hybrid logic instead of full LLM classification:
- improves reliability
- reduces hallucinations
- lowers cost

Trade-off:
- slightly less flexible

---

## Trade-Off 2 — Async Architecture
Using queues improves scalability.

Trade-off:
- added infrastructure complexity

---

## Trade-Off 3 — Modular Services
Separate layers improve maintainability.

Trade-off:
- more orchestration overhead

---

## Trade-Off 4 — Fallback Templates
Templates improve reliability during outages.

Trade-off:
- responses become less dynamic




# 15. Future Improvements

If given more time, I would extend this system with:

- Retrieval-Augmented Generation (RAG)
- Vector database memory
- Human-in-the-loop review
- AI analytics dashboard
- CRM behavioral feedback loops
- Multi-agent orchestration
- Real-time lead scoring updates

---

# Final Note

The focus of this submission was:
- practical engineering
- scalable architecture
- production awareness
- clear implementation thinking

Rather than unnecessary complexity.
